# main_pipeline (Kaggle)

Orchestrator chạy toàn bộ pipeline (build data -> train -> chat demo -> evaluate)
trên Kaggle (có GPU free).

**Chuẩn bị trước khi chạy:**
1. Upload cả thư mục `medquad_rag/` (chứa `src/`, `pipeline/`, `requirements.txt`)
   làm 1 Kaggle Dataset, rồi add vào notebook này (Add Input), HOẶC `!git clone`
   repo nếu đã đẩy lên GitHub.
2. Upload `medquad.json` vào `data/` của project (hoặc add làm Dataset riêng
   rồi copy vào `data/`).
3. Sửa biến `PROJECT_DIR` ở cell dưới cho đúng đường dẫn sau khi add input.

In [ ]:
pip install -q --force-reinstall --no-cache-dir "numpy==1.26.4" "pandas==2.2.2" "pyarrow>=14,<18"

In [1]:
!rm -rf medquad_rag
!git clone https://github.com/nguyentuanphat2594/test_llm_project

Cloning into 'test_llm_project'...
remote: Enumerating objects: 109, done.
remote: Counting objects: 100% (109/109), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 109 (delta 53), reused 101 (delta 46), pack-reused 0 (from 0)
Receiving objects: 100% (109/109), 2.84 MiB | 12.71 MiB/s, done.
Resolving deltas: 100% (53/53), done.


In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

In [3]:
PROJECT_DIR = "/kaggle/working/test_llm_project/medquad_rag"

import sys
sys.path.insert(0, PROJECT_DIR)

%cd /kaggle/working/test_llm_project/medquad_rag

/kaggle/working/test_llm_project/medquad_rag


In [4]:
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 95.8 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.9/503.9 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.9/511.9 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 32.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.8/494.8 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.7/175.7 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.4/396.4 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━

In [5]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
print("ragas import OK")

ragas import OK


## Bước 1: Build train.jsonl từ medquad.json

In [6]:
from pipeline import build_train_dataset
build_train_dataset.main()

Total samples (raw)   : 11548
Sample limit áp dụng  : 600
Hợp lệ (validated)    : 600
Bỏ qua (thiếu Q/A)    : 0
USE_RAG               : False
----------------------------------------
Train : 450 -> /kaggle/working/test_llm_project/medquad_rag/output/train.jsonl
Val   : 144 -> /kaggle/working/test_llm_project/medquad_rag/output/val.jsonl
Test  : 6 -> /kaggle/working/test_llm_project/medquad_rag/output/test.jsonl


## Bước 2: Train LoRA

In [7]:
from pipeline import train
train.main()

Loading model + tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,2.038000,2.239934


{'loss': 2.0122, 'grad_norm': 0.0, 'learning_rate': 0.0002, 'num_tokens': 6235.0, 'mean_token_accuracy': 0.563758447766304, 'epoch': 0.02}
{'loss': 2.1951, 'grad_norm': 0.0, 'learning_rate': 0.00019649122807017543, 'num_tokens': 10837.0, 'mean_token_accuracy': 0.5357726588845253, 'epoch': 0.04}
{'loss': 2.2582, 'grad_norm': 0.0, 'learning_rate': 0.00019298245614035088, 'num_tokens': 13966.0, 'mean_token_accuracy': 0.5262932181358337, 'epoch': 0.05}
{'loss': 1.8899, 'grad_norm': 0.0, 'learning_rate': 0.00018947368421052632, 'num_tokens': 19230.0, 'mean_token_accuracy': 0.608647495508194, 'epoch': 0.07}
{'loss': 2.1726, 'grad_norm': 0.0, 'learning_rate': 0.00018596491228070177, 'num_tokens': 22896.0, 'mean_token_accuracy': 0.5481975227594376, 'epoch': 0.09}
{'loss': 2.2177, 'grad_norm': 0.0, 'learning_rate': 0.0001824561403508772, 'num_tokens': 25955.0, 'mean_token_accuracy': 0.5410485714673996, 'epoch': 0.11}
{'loss': 1.9348, 'grad_norm': 0.0, 'learning_rate': 0.00017894736842105264, 'n

  0%|          | 0/72 [00:00<?, ?it/s]

{'eval_loss': 2.239933967590332, 'eval_runtime': 58.7569, 'eval_samples_per_second': 2.451, 'eval_steps_per_second': 1.225, 'eval_num_tokens': 252277.0, 'eval_mean_token_accuracy': 0.5509883711735407, 'epoch': 1.0}
{'train_runtime': 611.9442, 'train_samples_per_second': 0.735, 'train_steps_per_second': 0.093, 'train_loss': 2.098305260925962, 'epoch': 1.0}
Train xong. Lưu model vào /kaggle/working/test_llm_project/medquad_rag/output/output_model
HOÀN TẤT


## Bước 3: Demo chat (inference)

In [8]:
from pipeline import chat
chat.main()

Đang load model...
Load model xong.

CÂU HỎI: Triệu chứng của bệnh bạch cầu cấp ở người lớn là gì?
TRẢ LỜI: Triệu chứng của bệnh bạch cầu cấp ở người lớn bao gồm:

1. Tiredness: Người lớn có thể cảm thấy mình đang bị mất sức.
2. Easy bruising or bleeding: Có thể thấy vết thương dễ dàng xuất hiện hoặc chảy máu nhẹ hơn bình thường.
3. Weakness: Cảm giác nhạy cảm và khó khăn trong việc hoạt động.
4. Night sweats: Mệt mỏi khi ngủ.
5. Easy bruising: Đau đớn làm cho vết thương trở nên dễ tổn thương hơn.
6. Petechiae (bệnh nhân có dấu hiệu nhỏ của các nấm), chỉ số nhỏ nhưng rõ ràng.
7. Shortness of breath: Nên đi khám ngay lập tức nếu bạn cảm thấy khó thở.
8. Weight loss: Đất lượng giảm đáng kể.
9. Bone or stomach pain: Có thể đau骨, đau dạ dày, hay đau bụng.
10. Painful lumps: Các lỗ chân lông mạn tính có thể phát triển.

Đây đều là triệu chứng phổ biến mà bạn cần phải lưu ý vì họ có thể là dấu hiệu của một loạt các tình trạng khác nhau.


## Bước 4: Đánh giá bằng RAGAs

In [9]:
from pipeline import evaluate
evaluate.main()

Đang load model bị đánh giá (base + adapter LoRA vừa train)...
Đang tải dữ liệu đánh giá từ test.jsonl...
Số câu hỏi test: 6
Đang load model giám khảo (prometheus-eval/prometheus-7b-v2.0, tách biệt model vừa train)...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Đang chấm điểm bằng RAGAs (có thể chậm vì dùng model local)...
USE_RAG=False -> bỏ qua faithfulness/context_precision/context_recall (cần contexts thật, hiện không có). Chỉ chấm answer_relevancy.


Evaluating:   0%|          | 0/6 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[0]: OutOfMemoryError(CUDA out of memory. Tried to allocate 13.28 GiB. GPU 0 has a total capacity of 14.56 GiB of which 5.51 GiB is free. Including non-PyTorch memory, this process has 9.05 GiB memory in use. Of the allocated memory 7.37 GiB is allocated by PyTorch, and 1.48 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables))
ERROR:ragas.executor:Exception raised in Job[1]: OutOfMemoryError(CUDA out of memory. Tried to allocate 12.05 GiB. GPU 0 has a total capacity of 14.56 GiB of which 3.18 GiB is free. Including non-PyTorch memory, this process has 11.38 GiB memory in use. Of the allocated memory 10.81 GiB is allocated by PyTorch, and 376.03 MiB is reserved by PyTorch but unallocated. If reserved but unallocated m

KẾT QUẢ ĐÁNH GIÁ
{'answer_relevancy': nan}


## (Tuỳ chọn) Lưu output_model về lại Kaggle Output

`output/output_model/` đã nằm trong `/kaggle/working/`, Kaggle sẽ tự lưu làm
Output của notebook này khi Save Version — không cần thêm gì.

In [23]:
%cd /kaggle/working/test_llm_project/medquad_rag
!zip -r output.zip output

/kaggle/working/test_llm_project/medquad_rag
updating: output/ (stored 0%)
updating: output/test.jsonl (deflated 66%)
updating: output/val.jsonl (deflated 82%)
updating: output/.gitkeep (stored 0%)
updating: output/output_model/ (stored 0%)
updating: output/output_model/merges.txt (deflated 57%)
updating: output/output_model/adapter_config.json (deflated 56%)
updating: output/output_model/tokenizer_config.json (deflated 89%)
updating: output/output_model/adapter_model.safetensors (deflated 42%)
updating: output/output_model/chat_template.jinja (deflated 71%)
updating: output/output_model/vocab.json (deflated 61%)
updating: output/output_model/README.md (deflated 44%)
updating: output/output_model/tokenizer.json (deflated 81%)
updating: output/output_model/special_tokens_map.json (deflated 69%)
updating: output/output_model/training_args.bin (deflated 54%)
updating: output/output_model/added_tokens.json (deflated 67%)
updating: output/train.jsonl (deflated 83%)
